# Arboili Data Collection

This notebook downloads and validates all data sources for the project.

| Section | Data source | Module |
|---|---|---|
| 1 | Epidemiological data (SINAN + SIVEP) | R scripts |
| 2 | Google Trends — search index | `src/google_trends` |
| 3 | Google Trends — related topics & queries | `src/google_trends` |
| 4 | Climate data (INMET) | `src/climate` |
| 5 | Health bulletins (Boletins Epidemiológicos) | `src/bulletins` |
| 6 | EBC news articles (Agência Brasil) | `src/ebc` |
| 7 | Validation | all modules |

Run the cells in order. Each section is independent — skip any section whose data is already downloaded.

---
## 0 — Setup

In [1]:
import sys
from pathlib import Path

# Make src/ importable as a package root
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"

DIRS = {
    "epidemiological":       DATA_DIR / "epidemiological",
    "epidemiological/SINAN": DATA_DIR / "epidemiological" / "SINAN",
    "epidemiological/SIVEP": DATA_DIR / "epidemiological" / "SIVEP",
    "google_trends":         DATA_DIR / "google_trends",
    "climate":               DATA_DIR / "climate",
    "bulletins":             DATA_DIR / "bulletins",
    "news":                  DATA_DIR / "news",
}
for name, path in DIRS.items():
    path.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Output directories ready.")

PROJECT_ROOT: c:\Users\caioc\Documents\Arboili_datapaper
Output directories ready.


---
## 1 — Epidemiological Data (SINAN Dengue)

Downloads yearly dengue case CSVs from the Brazilian Ministry of Health S3 bucket via `src/epidemiological/sinan_dengue.py`.

Already-downloaded files are skipped automatically. A `manifest.csv` and `failures.csv` are written after every run so you can track what was retrieved.

To retry only failed years, call `extract()` again — skipped (existing) years will not be re-fetched.

> **Note:** Aggregation of the raw per-row CSVs into the final `SINAN_dengue_cases.csv.gz` is still done by the R script `r/scripts/2_transform_sinan_data.R`. Run it after this cell if you need the aggregated file.

In [ ]:
from src.epidemiological.sinan_dengue import extract

result = extract(
    out_dir=DATA_DIR / "epidemiological" / "SINAN",
    from_year=2010,
    to_year=2024,
)
print(result)

---
## 2 — Google Trends: Search Index

Downloads a 5-year rolling window of weekly search-interest values for each disease/symptom group × Brazilian state combination using pytrends (no API key required).

**Output:** `data/google_trends/GoogleTrends_search.csv`

The run takes a few minutes for the full dataset. If a 429 (rate-limit) error occurs, the script saves progress and stops — re-run the cell to continue from where it left off.

In [ ]:
from datetime import date
from src.google_trends.extract_gt_search import extract

result = extract(
    reference_date=date(2024, 12, 31),
    out_dir=DATA_DIR / "google_trends",
    popular_terms_path=DATA_DIR / "google_trends" / "popular_terms.csv",
    fu_path=DATA_DIR / "epidemiological" / "br_federative_units.csv",
)
print(result)

---
## 3 — Google Trends: Related Topics & Queries

Downloads monthly related topics and queries for the 4 disease groups × 28 Brazilian states from January 2020 onwards.

**Outputs:**
- `data/google_trends/GoogleTrends_related_topic.csv`
- `data/google_trends/GoogleTrends_related_query.csv`

> **Runtime warning:** ~13 400 requests at 5 s each ≈ 18 hours. The script saves progress on every 429 and can be resumed safely by re-running the cell.

In [ ]:
from src.google_trends.extract_gt_related import extract

result = extract(
    out_dir=DATA_DIR / "google_trends",
    fu_path=DATA_DIR / "epidemiological" / "br_federative_units.csv",
    popular_terms_path=DATA_DIR / "google_trends" / "popular_terms.csv",
)
print(result)

---
## 4 — Climate Data (INMET)

Downloads annual historical meteorological data from INMET (Instituto Nacional de Meteorologia) for every year from 2000 to the current year. Files already on disk are skipped.

**Output:** `data/climate/<year>.zip`

In [ ]:
import logging
from src.climate.download_inmet_data import extract

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", datefmt="%H:%M:%S")

result = extract(out_dir=DATA_DIR / "climate")
print(result)

---
## 5 — Health Bulletins (Boletins Epidemiológicos)

Downloads weekly epidemiological bulletins published by the Brazilian Ministério da Saúde from 2019 to the current year. Already-downloaded PDFs are skipped.

**Output:** `data/bulletins/<year>/<filename>.pdf`

In [ ]:
from src.bulletins.download_boletins import extract

result = extract(out_dir=DATA_DIR / "bulletins")
print(result)

---
## 6 — EBC News Articles (Agência Brasil)

Scrapes news articles from EBC's search engine. The scraper is resumable — re-running picks up from the last completed page and skips articles already in the manifest.

**Output:** `data/news/` (state.json, manifest.jsonl, listings/, articles/)

Add a cell per search query. Each query writes to its own subdirectory.

In [4]:
import argparse
from src.ebc.scraper import scrape

# --- Dengue ---
result = scrape(argparse.Namespace(
    query="dengue",
    site="agenciabrasil",
    types=["noticia", "pagina"],
    per_page=100,
    max_pages=10_000,
    output_dir=DATA_DIR / "news" / "dengue",
    delay=0.5
    ,
    restart=False,
    verbose=False,
))
print(result)

ExtractResult(downloaded=4720, existing=4720, failed=2, manifest=c:\Users\caioc\Documents\Arboili_datapaper\data\news\dengue\manifest.jsonl)


In [ ]:
'''# --- Chikungunya ---
scrape(argparse.Namespace(
    query="chikungunya",
    site="agenciabrasil",
    types=["noticia", "pagina"],
    per_page=100,
    max_pages=10_000,
    output_dir=DATA_DIR / "news" / "chikungunya",
    delay=1.0,
    restart=False,
    verbose=False,
))'''

---
## 7 — Validation

Smoke-tests for each module and a final inventory of all output files.

### 7.1 — Google Trends API helpers

In [ ]:
# Offline tests for gtrends helper functions (no network required)
from src.google_trends.gtrends_api import _to_timeframe, _time_label

cases = [
    ("2020-01", "2024-12"),
    ("2019-06", "2020-06"),
]
for start, end in cases:
    tf = _to_timeframe(start, end)
    label = _time_label(start, end)
    print(f"  {start} → {end}  |  timeframe={tf!r}  label={label!r}")

In [ ]:
# Live smoke test — one small GT request (requires internet)
try:
    from src.google_trends.gtrends_api import try_gtrends_api, _make_client
    client = _make_client()
    df, err = try_gtrends_api(
        topic_keyword="/m/09wsg",  # Dengue topic ID
        geo_location="BR",
        start_date="2024-01",
        end_date="2024-03",
        fun="graph",
        client=client,
        sleep=2,
    )
    if err:
        print(f"Error: {err}")
    else:
        print(f"GT smoke test OK — {len(df)} rows returned")
        print(df.head(3))
except Exception as exc:
    print(f"Skipped (network unavailable): {exc}")

### 7.2 — SINAN dengue URL helpers

In [ ]:
# URL construction test (offline)
from src.epidemiological.sinan_dengue import csv_zip_url

for year in [2010, 2020, 2024]:
    print(f"  {year}  →  {csv_zip_url(year)}")

In [ ]:
# S3 availability probe — checks which years are live (requires internet)
try:
    from src.epidemiological.sinan_dengue import probe_year
    available = [y for y in range(2010, 2027) if probe_year(y)]
    print("Available SINAN dengue years on S3:", available)
except Exception as exc:
    print(f"Skipped (network unavailable): {exc}")

### 7.3 — Output inventory

Final check: list every expected output file with its size and row count.

In [ ]:
import pandas as pd
from pathlib import Path

inventory = {
    "SINAN / manifest":                DATA_DIR / "epidemiological/SINAN/manifest.csv",
    "Google Trends / Search manifest": DATA_DIR / "google_trends/manifest_search.csv",
    "Google Trends / Related manifest":DATA_DIR / "google_trends/manifest_related.csv",
    "Google Trends / Search data":     DATA_DIR / "google_trends/GoogleTrends_search.csv",
    "Google Trends / Related topics":  DATA_DIR / "google_trends/GoogleTrends_related_topic.csv",
    "Google Trends / Related queries": DATA_DIR / "google_trends/GoogleTrends_related_query.csv",
    "Climate / manifest":              DATA_DIR / "climate/manifest.csv",
    "Bulletins / manifest":            DATA_DIR / "bulletins/manifest.csv",
    "News / dengue manifest":          DATA_DIR / "news/dengue/manifest.jsonl",
}

print(f"{'File':<45}  {'Size':>10}  {'Rows':>10}  {'Status summary'}")
print("-" * 90)
for label, path in inventory.items():
    if not path.exists():
        print(f"  {label:<43}  {'MISSING':>10}")
        continue
    size_kb = path.stat().st_size / 1024
    try:
        if path.suffix == ".jsonl":
            rows = sum(1 for _ in open(path, encoding="utf-8"))
            print(f"  {label:<43}  {size_kb:>9.1f}K  {rows:>10,}")
        else:
            df = pd.read_csv(path)
            if "status" in df.columns:
                summary = df["status"].value_counts().to_dict()
                print(f"  {label:<43}  {size_kb:>9.1f}K  {len(df):>10,}  {summary}")
            else:
                print(f"  {label:<43}  {size_kb:>9.1f}K  {len(df):>10,}")
    except Exception as e:
        print(f"  {label:<43}  {size_kb:>9.1f}K  (unreadable: {e})")